In [1]:
import pandas as pd
import statsmodels.api as sm

In [2]:
df = pd.read_csv(r"E:\data\Analysis Data\01_python_analysis\input\Supplementary_table_WT.csv")

In [3]:
("./Supplementary_table_WT.csv")
df = df.rename(columns={"Final pole_pole length (µm)": "Length"})
df = df[["Stage", "Cell", "Length"]].dropna()
df["Stage"] = df["Stage"].astype(str)
df["Cell"]  = df["Cell"].astype(str)
df["All"]   = 1  # single dummy group; we only want variance components


In [4]:
# Random intercepts for Stage and for Cell (no interaction term)
m = sm.MixedLM.from_formula(
    "Length ~ 1",
    groups="All",           # dummy grouping
    re_formula="0",
    vc_formula={"Stage": "0 + C(Stage)",
                "Cell":  "0 + C(Cell)"},
    data=df
)
res = m.fit(reml=True, method="lbfgs", maxiter=1000)
print(res.summary())

         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Length   
No. Observations: 745     Method:             REML     
No. Groups:       1       Scale:              0.5976   
Min. group size:  745     Log-Likelihood:     -966.9404
Max. group size:  745     Converged:          Yes      
Mean group size:  745.0                                
-------------------------------------------------------
              Coef.  Std.Err.   z   P>|z| [0.025 0.975]
-------------------------------------------------------
Intercept     14.275    1.736 8.222 0.000 10.872 17.677
Cell Var       0.355    0.087                          
Stage Var     18.012   14.791                          



In [5]:
# Extract variance components (Stage, Cell) and residual (measurement noise)
print("Residual variance (measurement noise):", float(res.scale))
# In recent statsmodels, the variance components live here:
print("Variance components:", getattr(res, "vcomp", "upgrade statsmodels to access res.vcomp reliably"))

Residual variance (measurement noise): 0.5976388679310705
Variance components: [ 0.3547546  18.01214269]
